# Debugging subplot layouts

When a composed figure looks wrong — wrong sizes, unexpected gaps, panels that should be joined but aren't — this notebook gives you a `show_layout(parent)` helper that overlays the layout decisions on top of the rendered SVG so you can see what plotlet actually allocated.


In [8]:
import plotlet as pt
import math

xs  = [i * 0.1 for i in range(64)]
sin = [math.sin(t) for t in xs]
cos = [math.cos(t) for t in xs]


## The `show_layout` helper

Each leaf is overlaid with two colored rectangles:

  * **Outer dashed rect** — the panel bbox the parent allocated (light fill).
  * **Inner solid rect** — the data area where artists draw (stronger fill).
  * **Light fill ring between them** — the panel's *margin* (ticks, labels, title go here).
  * **White slabs between leaves** — the *gaps* between bboxes (default 20 px, collapses to 0 for joined share-pair joints).

Helper reaches into `plotlet.layout` private internals — fine for an explainer / debug tool, not part of the public API.


In [9]:
from plotlet.layout import _measure, _allocate, _build_panel_opts, _effective_margin
from IPython.display import HTML

_PALETTE = ['#1f77b4', '#ff7f0e', '#d62728', '#2ca02c', '#9467bd', '#8c564b']

def show_layout(parent):
    panel_opts, _ = _build_panel_opts(parent)
    W, H = _measure(parent)
    W, H = int(round(W)), int(round(H))
    placements = []
    _allocate(parent, 0, 0, W, H, placements)
    overlay = []
    for i, (leaf, (x, y, w, h)) in enumerate(placements):
        col = _PALETTE[i % len(_PALETTE)]
        M = _effective_margin(leaf._fig._margin, panel_opts[id(leaf)], w, h)
        overlay.append(
            f'<rect x="{x}" y="{y}" width="{w}" height="{h}" '
            f'fill="{col}" fill-opacity="0.18" '
            f'stroke="{col}" stroke-opacity="0.6" stroke-dasharray="4,3"/>'
        )
        overlay.append(
            f'<rect x="{x + M["left"]}" y="{y + M["top"]}" '
            f'width="{w - M["left"] - M["right"]}" '
            f'height="{h - M["top"] - M["bottom"]}" '
            f'fill="{col}" fill-opacity="0.4"/>'
        )
    return HTML(parent.to_svg().replace('</svg>', ''.join(overlay) + '</svg>'))




## Examples — by complexity


### 1. Single plot

Panel bbox vs data area on a standalone chart — no shares, no composition. Useful for inspecting margins on a single chart before composing anything.


In [10]:
single = pt.chart(title='single', width=400, height=240, xlabel='x', ylabel='sin x')
single.line(xs, sin)
show_layout(single)


**How margins change with panel size.** Margins scale per-axis — `scale_w = min(1, panel_w / 600)` for left/right, `scale_h = min(1, panel_h / 400)` for top/bottom — but each side has a floor in `spec.size.margin_floor` it can't drop below. Two more sizes to see when floors kick in:


In [17]:
c = pt.chart(title='600 × 400 (spec default)', width=600, height=400, xlabel='x', ylabel='sin x')
c.line(xs, sin)
show_layout(c)


In [18]:
c = pt.chart(title='150 × 100', width=150, height=100, xlabel='x', ylabel='sin x')
c.line(xs, sin)
show_layout(c)


Reading the three side by side:

  - **600 × 400** — `scale = 1.0` on both axes. Every side uses its base from `spec.size.margin`: top=30, right=18, bottom=48, left=58. No floor active.
  - **400 × 240** (the original example) — `scale_h = 0.6` would shrink top to 18, but the floor of 24 catches it. Left, right, bottom scale cleanly.
  - **150 × 100** — `scale_w = scale_h = 0.25`. All four sides drop below their floors and float at floor values (top=24, right=4, bottom=20, left=20). The data area becomes a tiny strip in the center; most of the canvas is reserved margin.

Top hits its floor first (~320 px height) because a title takes a fixed amount of vertical space regardless of how small the panel is. Left, right, bottom hold up better because tick labels genuinely thin out as the data area shrinks.


### 2. Two-plot composition, no share

`(a | b) / c` with no `share_x=` / `share_y=`. The white band between two panels = the gap; the colored band inside each panel = its margin. `inner_gap` doesn't enter the picture — every panel keeps its full margin.


In [11]:
a = pt.chart(title='a', width=290, height=140); a.line(xs, sin)
b = pt.chart(title='b', width=290, height=140); b.line(xs, cos)
c = pt.chart(title='c', width=600, height=140)
c.line(xs, [s + co for s, co in zip(sin, cos)])
show_layout((a | b) / c)


### 3. Share-pair

Auto-zero-gap only fires for *sibling leaves* in the same parent. In `(a / c) | b`, `a` and `c` are siblings in a vertical and `c.share_x=a` collapses the gap between them. Both panels' inner margins on the joint shrink to `inner_gap`. `b` sits in its own column and is unaffected.


In [12]:
a = pt.chart(title='a', width=290, height=140); a.line(xs, sin)
c = pt.chart(title='c', width=290, height=140, share_x=a)
c.line(xs, [s + co for s, co in zip(sin, cos)])
b = pt.chart(title='b', width=290, height=280); b.line(xs, cos)
show_layout((a / c) | b)


### 4. Multi-share grid (ComplexHeatmap-flavored)

`top.share_x=main` (collapses vertically) and `tree.share_y=main` (collapses horizontally). `main` ends up with `inner_gap` on its top *and* left edges; `tree` gets it on its right; `top` gets it on its bottom.


In [13]:
main = pt.chart(title='main', width=440, height=260)
main.line([1, 2, 3, 4, 5], [2, 4, 1, 5, 3])
top  = pt.chart(title='top',  width=440, height=80,  share_x=main)
top.line([1, 2, 3, 4, 5], [1, 1, 3, 1, 1])
tree = pt.chart(title='tree', width=120, height=260, share_y=main)
tree.line([0, 1, 2], [2, 3, 4])
show_layout(pt.grid([
    [None, top ],
    [tree, main],
]))


## Tweaking layout constants

There's no public setter for `inner_gap` / `gap` yet, but `plotlet.layout` reads them as plain module-level names on each call. So you can rebind them at runtime to experiment — useful when sanity-checking what feels right for a given figure. (Restore originals at the end if you don't want global side-effects in the kernel.)


### Basic patch

Same `(a / c) | b` shape from §3 but with patched values — compare against §3's render to see the joint tighten and the b-side gap shrink.


In [14]:
import plotlet.layout as _l

def demo():
    a = pt.chart(title='a', width=290, height=140); a.line(xs, sin)
    c = pt.chart(title='c', width=290, height=140, share_x=a)
    c.line(xs, [s + co for s, co in zip(sin, cos)])
    b = pt.chart(title='b', width=290, height=280); b.line(xs, cos)
    return show_layout((a / c) | b)

orig_inner, orig_gap = _l._INNER_GAP, _l._GAP
try:
    _l._INNER_GAP = 5   # tighter share-pair joint
    _l._GAP       = 100   # tighter default spacing between non-shared panels
    out = demo()
finally:
    _l._INNER_GAP, _l._GAP = orig_inner, orig_gap
out


### On the grid example

Same patch applied to §4's ComplexHeatmap-flavored shape.


In [15]:
orig_inner, orig_gap = _l._INNER_GAP, _l._GAP
try:
    _l._INNER_GAP = 0
    _l._GAP       = 8
    main = pt.chart(title='main', width=440, height=260)
    main.line([1, 2, 3, 4, 5], [2, 4, 1, 5, 3])
    top  = pt.chart(title='top',  width=440, height=80,  share_x=main)
    top.line([1, 2, 3, 4, 5], [1, 1, 3, 1, 1])
    tree = pt.chart(title='tree', width=120, height=260, share_y=main)
    tree.line([0, 1, 2], [2, 3, 4])
    out = show_layout(pt.grid([
        [None, top ],
        [tree, main],
    ]))
finally:
    _l._INNER_GAP, _l._GAP = orig_inner, orig_gap
out


### Isolating `_GAP` (no-share layout)

A no-share layout has no joined joints, so `_INNER_GAP` doesn't enter the picture. Tweak `_GAP` here to see the inter-panel white slabs in isolation.


In [16]:
orig_inner, orig_gap = _l._INNER_GAP, _l._GAP
try:
    _l._GAP = 80   # try 0, 4, 20, 40 to see the effect
    a = pt.chart(title='a', width=290, height=140); a.line(xs, sin)
    b = pt.chart(title='b', width=290, height=140); b.line(xs, cos)
    c = pt.chart(title='c', width=600, height=140)
    c.line(xs, [s + co for s, co in zip(sin, cos)])
    out = show_layout((a | b) / c)
finally:
    _l._INNER_GAP, _l._GAP = orig_inner, orig_gap
out
